In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        continue

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# =====================================================
# 🔥 FULLY MAMBA MODULE 
# =====================================================
import types
import sys
import torch
import torch.nn.functional as F
import importlib.machinery

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

# Helper to create module with spec
def make_module(name):
    module = types.ModuleType(name)
    module.__spec__ = importlib.machinery.ModuleSpec(name, loader=None)
    return module

# Build structure
mamba_ssm = make_module("mamba_ssm")
ops = make_module("mamba_ssm.ops")
triton = make_module("mamba_ssm.ops.triton")
layernorm_gated = make_module("mamba_ssm.ops.triton.layernorm_gated")

# Attach function
layernorm_gated.rmsnorm_fn = _pure_rmsnorm_fn

# Link hierarchy
triton.layernorm_gated = layernorm_gated
ops.triton = triton
mamba_ssm.ops = ops

# Register modules
sys.modules["mamba_ssm"] = mamba_ssm
sys.modules["mamba_ssm.ops"] = ops
sys.modules["mamba_ssm.ops.triton"] = triton
sys.modules["mamba_ssm.ops.triton.layernorm_gated"] = layernorm_gated

print("✅ Fully patched mamba_ssm with __spec__")

✅ Fully patched mamba_ssm with __spec__


In [3]:
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages \ trl deepspeed

ERROR: Could not find a version that satisfies the requirement deepspeed (from versions: none)
ERROR: No matching distribution found for deepspeed


In [4]:
# =====================================================
# INSTALL (OFFLINE)
# =====================================================
!pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages \
transformers peft datasets accelerate trl

In [5]:
!pip install -q --no-index --find-links /kaggle/input/datasets/long2003/ariel-2025-mamba-casualconv1d \ trl

In [6]:
import transformers.utils.import_utils as imp_utils
imp_utils.is_mamba_2_ssm_available = lambda: False

In [7]:
import kagglehub

In [8]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")

In [9]:
import polars as pl
import torch
from datasets import Dataset

train_df = pl.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')

train_pd = train_df.to_pandas()
dataset = Dataset.from_pandas(train_pd)

print(dataset[0])

{'id': '00066667', 'prompt': "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100", 'answer': '10010111'}


In [10]:
def format_prompt(example):
    q = example["prompt"]
    a = str(example["answer"]).strip()

    return {
        "text": f"""
You are a world-class mathematical reasoning engine.

Solve step by step.
- Break into logical steps
- Verify each step
- If mistake, correct it
- Think carefully before final answer

Question:
{q}

Solution:
{a}

Final Answer: \\boxed{{{a}}}
"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

In [11]:
def generate_synthetic(example):
    q = example["prompt"]

    synthetic_q = f"""
Modify this problem to make it harder but solvable:

Original:
{q}

Hard Version:
"""

    return {"synthetic_prompt": synthetic_q}

dataset = dataset.map(generate_synthetic)

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

In [13]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

In [14]:
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType

In [15]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()

In [16]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [17]:
import os
os.makedirs("/kaggle/working/offload", exist_ok=True)

In [18]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

print("Model loaded successfully.")

The fast path is not available because on of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.


In [19]:
model.gradient_checkpointing_enable()

In [20]:

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [21]:
model.train()
#trainer.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): NemotronHForCausalLM(
      (backbone): NemotronHModel(
        (embeddings): Embedding(131072, 2688)
        (layers): ModuleList(
          (0): NemotronHBlock(
            (norm): NemotronHRMSNorm()
            (mixer): NemotronHMamba2Mixer(
              (act): SiLUActivation()
              (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144)
              (in_proj): lora.Linear(
                (base_layer): Linear(in_features=2688, out_features=10304, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2688, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=10304, bias=False)
                )
                (lora_embedd

In [22]:
import re
from collections import Counter

def extract_boxed(text):
    match = re.findall(r'\\boxed{(.*?)}', text)
    return match[-1] if match else None


def generate_once(question, temp=0.7):
    prompt = f"""
Solve step by step. Verify reasoning.
Final answer must be inside \\boxed{{}}.

Question:
{question}

Solution:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=temp,
        top_p=0.9,
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


def self_consistency(question, n=5):
    answers = []

    for _ in range(n):
        out = generate_once(question)
        ans = extract_boxed(out)
        if ans:
            answers.append(ans)

    if not answers:
        return "0"

    return Counter(answers).most_common(1)[0][0]

In [23]:
def refine_answer(question, initial_answer):
    prompt = f"""
A model solved this problem but may be wrong.

Question:
{question}

Proposed Answer:
{initial_answer}

Check reasoning carefully and correct if needed.
Return final answer in \\boxed{{}}.
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.0,
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [24]:
def final_solver(question):
    # Step 1: multiple reasoning paths
    best = self_consistency(question, n=5)

    # Step 2: refine answer
    refined = refine_answer(question, best)

    final = extract_boxed(refined)

    return final if final else best

In [25]:
model.save_pretrained("/kaggle/working")

import subprocess
subprocess.run("zip -r submission.zip /kaggle/working", shell=True)

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/offload/ (stored 0%)
  adding: kaggle/working/adapter_config.json (deflated 55%)
  adding: kaggle/working/README.md (deflated 66%)
  adding: kaggle/working/__notebook__.ipynb (deflated 95%)
  adding: kaggle/working/adapter_model.safetensors (deflated 77%)


CompletedProcess(args='zip -r submission.zip /kaggle/working', returncode=0)